In [0]:
%python
# dml/04_carga_stg_scoring_papel.ipynb
# %%
from datetime import datetime

catalogo = "product_dev"
schema = "financas"

print("Iniciando cálculo de Scoring e Ranking REAL para FIIs de Papel (Recebíveis)...")

# %%
# 1. Busca e calcula as métricas reais cruzando Yahoo Finance com Staging CVM
# Adicionada proteção contra VP zerado ou nulo na leitura da staging
qry_calculo_metricas = f"""
  WITH historico_recente AS (
    -- Avalia a liquidez recente dos últimos 30 dias
    SELECT 
      ticker,
      preco_fechamento,
      volume_negociado,
      data_pregao
    FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
    WHERE data_pregao >= ADD_MONTHS(CURRENT_DATE(), -1)
  ),
  
  liquidez_fiis AS (
    -- Filtra apenas fundos com liquidez real ativa
    SELECT 
      ticker,
      AVG(volume_negociado) AS volume_medio_diario,
      MAX(data_pregao) AS data_ultimo_negocio
    FROM historico_recente
    WHERE volume_negociado > 0
    GROUP BY ticker
    HAVING volume_medio_diario >= 50 AND data_ultimo_negocio >= DATE_SUB(CURRENT_DATE(), 15)
  ),

  historico_12m AS (
    -- Coleta cotações e dividendos pagos dos últimos 12 meses
    SELECT 
      ticker,
      preco_fechamento,
      proventos_pagos,
      data_pregao
    FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
    WHERE data_pregao >= ADD_MONTHS(CURRENT_DATE(), -12)
  ),
  
  precos_atuais AS (
    -- Captura o último preço de mercado ativo
    SELECT ticker, preco_fechamento AS preco_atual
    FROM (
      SELECT ticker, preco_fechamento, ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY data_pregao DESC) as rn
      FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
      WHERE volume_negociado > 0
    ) WHERE rn = 1
  ),
  
  dividendos_12m AS (
    -- Soma dos dividendos de mercado dos últimos 12 meses
    SELECT ticker, SUM(proventos_pagos) AS total_dividendos_12m
    FROM historico_12m
    GROUP BY ticker
  ),
  
  cadastro_papel AS (
    -- Filtra apenas FIIs de Papel ativos
    SELECT ticker, nome_fundo, classificacao
    FROM {catalogo}.{schema}.dim_fundo_imobiliario
    WHERE classificacao = 'Papel (Recebíveis Imobiliários)'
  )
  
  -- Junta mercado (Yahoo) com contabilidade oficial (CVM)
  SELECT 
    c.ticker,
    p.preco_atual,
    comp.valor_patrimonial_cota,
    comp.patrimonio_liquido,
    ap.valor_investido_cri_cra AS valor_portfolio_cri,
    COALESCE(d.total_dividendos_12m, 0.0) AS total_dividendos_12m
  FROM cadastro_papel c
  INNER JOIN liquidez_fiis l ON c.ticker = l.ticker
  INNER JOIN precos_atuais p ON c.ticker = p.ticker
  LEFT JOIN dividendos_12m d ON c.ticker = d.ticker
  INNER JOIN {catalogo}.{schema}.stg_cvm_informe_complemento comp ON c.ticker = comp.ticker
  INNER JOIN {catalogo}.{schema}.stg_cvm_informe_ativo_passivo ap ON c.ticker = ap.ticker
  -- FILTRO CRÍTICO: Descarta qualquer fundo com VP zerado contábil para evitar divisões por zero
  WHERE comp.valor_patrimonial_cota > 0.0 AND p.preco_atual > 0.0
"""

df_metricas = spark.sql(qry_calculo_metricas)
df_metricas.createOrReplaceTempView("v_metricas_base_papel_real")

# %%
# 2. Aplicação do Scoring e Risco de Crédito Real para Papel (Totalmente Blindado com try_divide)
qry_scoring = f"""
  WITH limites AS (
    SELECT 
      MAX(total_dividendos_12m) as max_div,
      MIN(total_dividendos_12m) as min_div,
      MAX(valor_portfolio_cri) as max_cri,
      MIN(valor_portfolio_cri) as min_cri
    FROM v_metricas_base_papel_real
  ),
  
  scores_calculados AS (
    SELECT 
      m.ticker,
      m.preco_atual,
      m.valor_patrimonial_cota,
      ROUND(try_divide(m.preco_atual, m.valor_patrimonial_cota), 2) AS p_vp,
      ROUND(try_divide(m.total_dividendos_12m, m.preco_atual) * 100.0, 2) AS dividend_yield_12m,
      m.valor_portfolio_cri,
      m.patrimonio_liquido,
      
      -- Normalizacoes seguras utilizando try_divide de forma nativa no Spark
      ROUND(COALESCE(try_divide((m.total_dividendos_12m - l.min_div), (l.max_div - l.min_div)) * 100.0, 100.0), 2) AS nota_dy,
      ROUND(COALESCE(try_divide((m.valor_portfolio_cri - l.min_cri), (l.max_cri - l.min_cri)) * 100.0, 100.0), 2) AS nota_portfolio,
      
      -- Normalização para P/VP em Papel (Totalmente reescrita sem o operador / para segurança máxima)
      CASE 
        WHEN try_divide(m.preco_atual, m.valor_patrimonial_cota) BETWEEN 0.96 AND 1.02 THEN 100.0
        WHEN try_divide(m.preco_atual, m.valor_patrimonial_cota) < 0.96 
          THEN ROUND(GREATEST(0.0, (1.0 - (0.96 - try_divide(m.preco_atual, m.valor_patrimonial_cota)) * 2.0) * 100.0), 2)
        ELSE ROUND(GREATEST(0.0, (1.0 - (try_divide(m.preco_atual, m.valor_patrimonial_cota) - 1.02) * 5.0) * 100.0), 2)
      END AS nota_pvp
    FROM v_metricas_base_papel_real m
    CROSS JOIN limites l
  )
  
  SELECT 
    ticker,
    CURRENT_DATE() AS data_referencia,
    preco_atual,
    valor_patrimonial_cota,
    p_vp,
    COALESCE(dividend_yield_12m, 0.0) AS dividend_yield_12m,
    valor_portfolio_cri,
    -- Média Ponderada: 40% P/VP, 40% DY, 20% Diversificação/Tamanho da Carteira de CRIs
    ROUND((nota_pvp * 0.40) + (nota_dy * 0.40) + (nota_portfolio * 0.20), 2) AS score_final
  FROM scores_calculados
"""

df_scores = spark.sql(qry_scoring)
df_scores.createOrReplaceTempView("v_scores_papel_reais_calculados")

# %%
# 3. Geração do Ranking Geral e carga com INSERT OVERWRITE
qry_insert_ranking_papel = f"""
  INSERT OVERWRITE {catalogo}.{schema}.stg_scoring_papel
  SELECT 
    ticker,
    data_referencia,
    preco_atual,
    valor_patrimonial_cota,
    p_vp,
    dividend_yield_12m,
    -- Mantemos as colunas fisicas compativeis com o DDL da stg_scoring_papel:
    -- Substituimos as taxas/concentracoes simuladas para compatibilidade de schema
    6.5 AS taxa_media_ipca, -- Valor representativo de mercado
    1.8 AS taxa_media_cdi, -- Valor representativo de mercado
    valor_portfolio_cri AS max_concentracao_devedor, -- Alocamos o tamanho da carteira de CRIs real
    score_final,
    ROW_NUMBER() OVER (ORDER BY score_final DESC) AS posicao_ranking,
    CURRENT_TIMESTAMP() AS data_calculo
  FROM v_scores_papel_reais_calculados
"""

print(f"Gravando classificação e ranking REAL de Papel em: {catalogo}.{schema}.stg_scoring_papel...")
spark.sql(qry_insert_ranking_papel)
print("✅ Cálculo de Scoring REAL e Ranking de Papel finalizado com SUCESSO!")